# Model Training

Document model decisions and test training snippets

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 04/03/2026   | Martin | Created   | Notebook created. Describe metrics selected | 

# Content

* [Introduction](#introduction)
* [Metrics Used](#metrics-used)
* [Data Loading](#data-loading)

# Introduction

Board game recommendation system using the two-tower architecture with a reranker.

# Metrics Used

- __Recall@K__ - Information retrieval and recommendation metrics that measures the proportion of relevant items found in the top-K results. Results here are not rank aware i.e does not matter which position in the top-K the item was found in, as long as it's inside there
$$
\frac{Relevant\ Items\ in\ Top\ K}{Total\ Relevant\ Items}
$$

- __Normalized Discounted Cumulative Gains (NDCG@K)__ - Measures the gain of an item based on its relevantce grade and position in the top-K results. It rewards placing highly relevant items at the top of the list
  * $rel_i$: Relevance score (e.g whether the item showed up in the top-K spots)
$$
NDCG@K = \frac{DCG@K}{IDCG@K} \\
DCG@K = \sum_{k=1}^{K}\frac{rei_i}{log_2(i+1)} \\
IDCG@K = Maximum\ achievable\ DCG
$$
- __Mean Reciprocal Rank (MRR)__ - Ranking quality metrics that considers the position of the first relevant item in the ranked list. Range from 0-1 where "1" is the first relevant item being at the top of the list
  * $U$: Total number of users
$$
MRR = \frac{1}{U}\sum_{u=1}{U}\frac{1}{rank_i}
$$
- __HitRate@K__ - Measures the number of users whom at least one relevant item is present in the top-K list. Coarse metric
- __Mean Average Precision (MAP@K)__ - Ranking quality metric that considers the number of relevant recommendations and their position in the list.
  * $N$: Total number of relevant items in the top-K results
  * Divide the total number of relevant items by the total number of items until this position
$$
MAP@K = \frac{1}{U}\sum_{u=1}^U AP@K_u \\
AP@K = \frac{1}{N}\sum_{k=1}^K Precision(k) \times rel(k)
$$

- __Coverage@K__ - Measures the percentage of unique items that was recommended over the total number of items. Evalutes the diversity of items
$$
Coverage@K = \frac{Number\ of\ unique\ items\ recommended\ in\ top-K}{Total\ number\ of\ items\ in\ catalog}
$$

# Data Loading

## Converting to DuckDB calls

Original code uses CSVs, converting to DuckDB calls

In [11]:
import duckdb

In [12]:
DUCKDB_PATH = "../data/bgg_db.duckdb"

In [13]:
con = duckdb.connect(DUCKDB_PATH)

In [14]:
con.sql("SHOW ALL TABLES")

┌──────────┬─────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┐
│ database │ schema  │   name   │                                                                                                                                                             column_names                                                                                                                                                       

In [15]:
games = con.sql("SELECT * FROM bgg.games").fetchdf()
comments = con.sql("SELECT * FROM bgg.comments").df()

In [17]:
games.columns

Index(['id', 'name', 'description', 'year_published', 'min_players',
       'max_players', 'suggested_players', 'playing_time', 'min_playing_time',
       'max_playing_time', 'min_age', 'suggested_age', 'category_ids',
       'categories', 'mechanic_ids', 'mechanics', 'family_ids', 'families',
       'designer_ids', 'designers', 'artist_ids', 'artists', 'num_ratings',
       'rating', 'bayes_rating', 'complexity'],
      dtype='str')

## Testing Supabase calls

In [8]:
import os
from dotenv import load_dotenv
from supabase import create_client, Client
import pandas as pd
load_dotenv()

True

In [3]:
url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(url, key)

In [5]:
response = (
  supabase.table("comp_game_log")
  .select("*")
  .execute()
)

In [9]:
pd.DataFrame(response.data)

,id,session_id,created_at,date_played,game_id,game_title,game_weight,game_length,num_players,profile_id,...,win_contrib,score,high_score,quarter,month,year,is_tie,is_first_play,rating,created_by
0,175,92697a9182a3,2026-01-24T08:46:49.863284+00:00,2026-01-24,118048,Targi,2.3390,60,2,10,...,100,4.33649,False,1,0,2026,False,True,NaN,10
1,176,92697a9182a3,2026-01-24T08:46:49.863284+00:00,2026-01-24,118048,Targi,2.3390,60,2,11,...,0,1.08412,False,1,0,2026,False,True,NaN,10
2,196,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,11,...,250,5.95074,False,1,1,2026,True,False,NaN,11
3,197,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,21,...,0,1.48768,False,1,1,2026,True,True,NaN,11
4,198,adbb1193fc22,2026-02-03T14:45:06.755792+00:00,2026-02-03,393429,Critter Kitchen,2.4444,60,5,22,...,0,0.66119,False,1,1,2026,False,True,NaN,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293,304,5c4c9341c0f8,2026-04-04T09:36:59.810151+00:00,2026-04-04,338376,A Gest of Robin Hood,3.0357,90,2,20,...,100,5.12234,False,2,3,2026,False,True,NaN,10
294,305,5c4c9341c0f8,2026-04-04T09:36:59.810151+00:00,2026-04-04,338376,A Gest of Robin Hood,3.0357,90,2,10,...,0,1.28059,False,2,3,2026,False,True,NaN,10
295,306,f4faa4d09d04,2026-04-04T10:41:34.352216+00:00,2026-04-04,183251,Karuba,1.4234,40,3,20,...,150,3.96175,False,2,3,2026,False,False,NaN,10
296,307,f4faa4d09d04,2026-04-04T10:41:34.352216+00:00,2026-04-04,183251,Karuba,1.4234,40,3,27,...,0,0.99044,False,2,3,2026,False,True,NaN,10


In [1]:
import sys
sys.path.append('../src')

In [ ]:
%load_ext watermark
%watermark